# Visualize static–dynamic split-half RSA results

This notebook visualizes the outputs of `run_static_dynamic_split_half_rsa.py`. It does not reload the large raster files or recompute RDMs. The key comparison is between within-condition self-consistency and the amount of dynamic RDM variance explained by the static condition.

In [3]:
from dataclasses import dataclass
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "config.yaml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find config.yaml above the notebook directory.")
    # end if PROJECT_ROOT.parent == PROJECT_ROOT
    PROJECT_ROOT = PROJECT_ROOT.parent
# end while not config.yaml exists

@dataclass
class Cfg:
    results_dir: Path | None = None
    figure_dpi: int = 180
    show_split_repeat_uncertainty: bool = True

cfg = Cfg()
if cfg.results_dir is None:
    cfg.results_dir = (
        PROJECT_ROOT / "results" / "static_dynamic_split_half_rsa"
        / "baby1_260716to24_vs_baby1_260718to27"
    )
# end if cfg.results_dir is None
cfg.results_dir = Path(cfg.results_dir).expanduser()
data_path = cfg.results_dir / "split_half_static_dynamic_rsa.npz"
config_path = cfg.results_dir / "config.json"
if not data_path.is_file():
    raise FileNotFoundError(
        f"Could not find {data_path}. Run run_static_dynamic_split_half_rsa.py first, "
        "or set cfg.results_dir to an existing result directory."
    )
# end if not data_path.is_file

results = dict(np.load(data_path, allow_pickle=False))
saved_config = {}
if config_path.is_file():
    with open(config_path, "r") as file:
        saved_config = json.load(file)
    # end with open
# end if config_path.is_file
print(f"Loaded {data_path}")
print(f"Shared stimuli: {len(results['shared_stimuli'])}")
print(f"Saved analysis configuration: {saved_config}")


FileNotFoundError: Could not find /Users/tizianocausin/Desktop/static_dynamic/results/static_dynamic_split_half_rsa/baby1_260716to24_vs_baby1_260718to27/split_half_static_dynamic_rsa.npz. Run run_static_dynamic_split_half_rsa.py first, or set cfg.results_dir to an existing result directory.

## Time-resolved self-consistency

Raw consistency correlates the channel × stimulus response vectors from two repetition halves. RDM consistency correlates the corresponding stimulus RDMs. Shaded regions show the 95% empirical interval over random split repetitions.

In [4]:
def plot_split_half_timecourses(results, cfg):
    """Plot raw and RDM split-half consistency for both conditions."""
    figure, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=False)
    condition_specs = (
        ("static", "tab:blue", results["static_times_ms"]),
        ("dynamic", "tab:orange", results["dynamic_times_ms"]),
    )
    for condition, color, times_ms in condition_specs:
        raw_values = results[f"{condition}_raw_split_half"]
        rdm_values = results[f"{condition}_rdm_split_half"]
        axes[0].plot(times_ms, np.nanmean(raw_values, axis=0), color=color, label=condition)
        axes[1].plot(times_ms, np.nanmean(rdm_values, axis=0), color=color, label=condition)
        if cfg.show_split_repeat_uncertainty and raw_values.shape[0] > 1:
            axes[0].fill_between(
                times_ms, np.nanpercentile(raw_values, 2.5, axis=0),
                np.nanpercentile(raw_values, 97.5, axis=0), color=color, alpha=0.18,
            )
            axes[1].fill_between(
                times_ms, np.nanpercentile(rdm_values, 2.5, axis=0),
                np.nanpercentile(rdm_values, 97.5, axis=0), color=color, alpha=0.18,
            )
        # end if cfg.show_split_repeat_uncertainty
    # end for condition
    axes[0].set(title="Raw response split-half consistency", xlabel="Time from onset (ms)", ylabel="Correlation")
    axes[1].set(title="RDM split-half consistency", xlabel="Time from onset (ms)", ylabel="RDM correlation")
    for axis in axes:
        axis.axhline(0, color="0.6", linewidth=0.8)
        axis.legend()
        axis.grid(alpha=0.2)
    # end for axis
    figure.tight_layout()
    return figure
# EOF

figure = plot_split_half_timecourses(results, cfg)
plt.show()


NameError: name 'results' is not defined

## Static–dynamic RDM explanation

The matrix contains the cross-temporal RDM correlation between dynamic and static responses. Squaring it gives the descriptive fraction of dynamic RDM variance associated with each static timepoint. The right panel shows the maximum static (r^2) within the configured static reference window.

In [5]:
dynamic_times_ms = results["dynamic_times_ms"]
static_times_ms = results["static_times_ms"]
explained_variance = results["maximum_static_explained_variance"]

figure, axes = plt.subplots(1, 2, figsize=(14, 5))
image = axes[0].imshow(
    results["static_dynamic_rdm_similarity"] ** 2,
    origin="lower", aspect="auto", cmap="magma", vmin=0, vmax=1,
    extent=(static_times_ms[0], static_times_ms[-1], dynamic_times_ms[0], dynamic_times_ms[-1]),
)
axes[0].set(title="Static-explained dynamic RDM variance", xlabel="Static time (ms)", ylabel="Dynamic time (ms)")
figure.colorbar(image, ax=axes[0], label="Squared RDM correlation")
axes[1].plot(dynamic_times_ms, explained_variance, color="black", linewidth=2)
axes[1].set(title="Maximum static explanation", xlabel="Dynamic time (ms)", ylabel="Maximum static RDM $r^2$", ylim=(-0.05, 1.05))
axes[1].grid(alpha=0.2)
figure.tight_layout()
plt.show()


NameError: name 'results' is not defined

## Explanation versus consistency

Each point is one dynamic timepoint. The color encodes dynamic time, allowing temporal regions with high static explanation to be compared with their within-condition raw and RDM reliability.

In [6]:
dynamic_raw_consistency = np.nanmean(results["dynamic_raw_split_half"], axis=0)
dynamic_rdm_consistency = np.nanmean(results["dynamic_rdm_split_half"], axis=0)

figure, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for axis, consistency, xlabel in zip(
        axes,
        (dynamic_raw_consistency, dynamic_rdm_consistency),
        ("Dynamic raw split-half correlation", "Dynamic RDM split-half correlation"),
        ):
    scatter = axis.scatter(
        consistency, explained_variance, c=dynamic_times_ms,
        cmap="viridis", s=28, alpha=0.85,
    )
    axis.set(xlabel=xlabel, ylabel="Static-explained dynamic RDM variance ($r^2$)")
    axis.axhline(0, color="0.7", linewidth=0.8)
    axis.axvline(0, color="0.7", linewidth=0.8)
    axis.grid(alpha=0.2)
# end for axis, consistency, xlabel
figure.colorbar(scatter, ax=axes, label="Dynamic time from onset (ms)")
figure.suptitle("Static explanation versus dynamic self-consistency")
figure.tight_layout()
plt.show()


NameError: name 'results' is not defined

## Optional numerical summary

This reports the dynamic timepoint with the largest static explanation and the corresponding raw/RDM split-half consistency.

In [2]:
best_index = int(np.nanargmax(explained_variance))
print(f"Best static explanation: {explained_variance[best_index]:.3f} at dynamic time {dynamic_times_ms[best_index]:.1f} ms")
print(f"Raw dynamic split-half consistency there: {dynamic_raw_consistency[best_index]:.3f}")
print(f"RDM dynamic split-half consistency there: {dynamic_rdm_consistency[best_index]:.3f}")


NameError: name 'np' is not defined